# Week 4 · Try It 3 (at home): YOLO on a Drone ROS Bag

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weisongwen/AAE4011-S22627/blob/main/Lecture%20Notes/Week%204/yolo_ros_bag.ipynb)

AAE4011 Artificial Intelligence in Unmanned Autonomous Systems · Semester 2, 2026/27

The bag `uav_intersection.bag` holds 23 seconds of drone video over a busy junction: one `sensor_msgs/CompressedImage` topic, 1280 × 720, 10 Hz. It was packed from one sequence of the public VisDrone2019-MOT dataset (Zhu et al., IEEE TPAMI 2021; academic use only), whose hand-labelled boxes are in `uav_intersection_gt.csv`.

No ROS installation is needed: the `rosbags` package reads ROS 1 bags in pure Python. The same detector as a real ROS node is `yolo_node.py` (lecture slides, Appendix B).

**Before you start**
1. Download `uav_intersection.bag` (41 MB) and `uav_intersection_gt.csv` (1.3 MB) from Blackboard (Week 4). They are not on GitHub: VisDrone is for academic use only.
2. *Runtime → Change runtime type → GPU* makes Parts 3 and 4 fast; a CPU runtime also works (a few minutes).

**Questions** (the same as on the slide)
- (a) How many frames are in the bag, on which topic, at what rate?
- (b) How many cars and people does YOLO11n find in frame 117 (confidence 0.25)? How many are there?
- (c) What are the precision and recall over the whole bag at confidence 0.25, and then at 0.1?
- (d) Repeat at `imgsz=1280`. What do you gain, and what does it cost?

In [ ]:
%pip install -q ultralytics rosbags

Upload the two files from Blackboard: run the next cell and pick both files in the dialog (or drag them into the Files panel on the left). Colab deletes uploaded files when the runtime ends.

In [ ]:
import os
FILES = ("uav_intersection.bag", "uav_intersection_gt.csv")
missing = [f for f in FILES if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files          # in Colab a file picker opens below this cell
        print("choose:", ", ".join(missing))
        files.upload()
    except ImportError:
        print("put these files from Blackboard next to this notebook:", ", ".join(missing))
print({f: os.path.getsize(f) // 1024 for f in FILES if os.path.exists(f)}, "KB")

## Part 1 · What is in the bag? (like `rosbag info`)

In [ ]:
from rosbags.rosbag1 import Reader

with Reader("uav_intersection.bag") as bag:
    dur = bag.duration / 1e9                       # nanoseconds -> seconds
    for c in bag.connections:
        print(f"{c.topic}  {c.msgtype}  {c.msgcount} messages")
        print(f"duration {dur:.1f} s, rate {(c.msgcount - 1) / dur:.1f} Hz")

## Part 2 · Read every frame, detect, and look at frame 117

In [ ]:
import cv2, csv, numpy as np
from collections import Counter
from rosbags.typesys import Stores, get_typestore

ts = get_typestore(Stores.ROS1_NOETIC)
jpg = {}                                           # seq -> JPEG bytes (decoded when needed)
with Reader("uav_intersection.bag") as bag:
    conns = [c for c in bag.connections if c.topic == "/camera/image_raw/compressed"]
    for conn, t, raw in bag.messages(connections=conns):
        msg = ts.deserialize_ros1(raw, conn.msgtype)
        jpg[msg.header.seq] = np.asarray(msg.data, np.uint8).copy()

def frame(seq):
    return cv2.imdecode(jpg[seq], cv2.IMREAD_COLOR)   # BGR image, 720 x 1280 x 3

gt = {}                                            # seq -> [(class, [x1, y1, x2, y2]), ...]
with open("uav_intersection_gt.csv", newline="") as f:
    for row in csv.DictReader(f):
        box = [float(row[c]) for c in ("x1", "y1", "x2", "y2")]
        gt.setdefault(int(row["frame"]), []).append((row["coco_class"], box))
print(len(jpg), "frames, first", min(jpg), "last", max(jpg), " size", frame(0).shape)

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

model = YOLO("yolo11n.pt")                        # downloads 5.6 MB the first time
r = model.predict(frame(117), conf=0.25, verbose=False)[0]
print("YOLO11n, frame 117:     ", dict(Counter(r.names[int(c)] for c in r.boxes.cls.tolist())))
print("ground truth, frame 117:", dict(Counter(c for c, _ in gt[117] if c not in ("ignore", "other"))))

plt.figure(figsize=(12, 7))
plt.imshow(r.plot(line_width=1, font_size=8)[:, :, ::-1])   # BGR -> RGB
plt.axis("off"); plt.show()

## Part 3 · Score the whole bag: precision and recall

A detection is correct (TP) if its IoU with a not-yet-matched true box of the same class is at least 0.5; detections are taken in order of confidence. Unmatched detections are FP and unmatched true boxes FN. Detections in VisDrone's 'ignored regions', or on objects COCO has no class for (tricycles), are not counted.

In [ ]:
import time

CLASSES = ("person", "bicycle", "car", "motorcycle", "bus", "truck")

def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def match_frame(dets, gts, conf):
    """TP, FP, FN of one frame, summed over the classes."""
    ign = [b for c, b in gts if c == "ignore"]
    other = [b for c, b in gts if c == "other"]
    tp = fp = fn = 0
    for cls in CLASSES:
        g = [b for c, b in gts if c == cls]
        used = [False] * len(g)
        for _, _, b in sorted((d for d in dets if d[0] == cls and d[1] >= conf), key=lambda d: -d[1]):
            cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
            if any(r[0] <= cx <= r[2] and r[1] <= cy <= r[3] for r in ign):
                continue
            best, j = 0.0, -1
            for k, gb in enumerate(g):
                if not used[k] and iou(b, gb) > best:
                    best, j = iou(b, gb), k
            if best >= 0.5:
                used[j] = True
                tp += 1
            elif not any(iou(b, ob) >= 0.5 for ob in other):
                fp += 1
        fn += used.count(False)
    return tp, fp, fn

def run_all(imgsz):
    """Detect on every frame (down to confidence 0.05) and time the network."""
    model.predict(frame(0), imgsz=imgsz, verbose=False)            # warm-up
    dets, ms = {}, []
    for seq in sorted(jpg):
        img = frame(seq)
        t = time.perf_counter()
        r = model.predict(img, imgsz=imgsz, conf=0.05, verbose=False)[0]
        ms.append((time.perf_counter() - t) * 1000)
        dets[seq] = [(r.names[int(c)], p, b) for b, c, p in
                     zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist())]
    return dets, float(np.median(ms))

def score(dets, conf):
    tp, fp, fn = np.sum([match_frame(d, gt.get(seq, []), conf) for seq, d in dets.items()], axis=0)
    return tp / (tp + fp), tp / (tp + fn)

det640, ms640 = run_all(640)
for conf in (0.25, 0.1):
    P, R = score(det640, conf)
    print(f"imgsz 640, conf {conf}: precision {P:.3f}, recall {R:.3f}")
print(f"median {ms640:.1f} ms per frame")

**Question (c).** Why does recall go up and precision go down when the threshold drops? Which matters more for a drone that must not fly into people?

## Part 4 · A larger input image

In [ ]:
det1280, ms1280 = run_all(1280)
P, R = score(det1280, 0.25)
print(f"imgsz 1280, conf 0.25: precision {P:.3f}, recall {R:.3f}")
print(f"median {ms1280:.1f} ms per frame, {ms1280 / ms640:.1f} times slower than at 640")

---
## Check your answers (try first!)

- (a) 233 frames on `/camera/image_raw/compressed`, 23.2 s, 10 Hz.
- (b) 18 cars and 7 people (plus a motorcycle and a traffic light); the ground truth has 33 cars and 42 people. Most people here are only 10 to 20 pixels tall after the frame is shrunk to 640.
- (c) Confidence 0.25: precision 0.867, recall 0.240. Confidence 0.1: precision 0.631, recall 0.374.
- (d) imgsz 1280 at 0.25: precision 0.694, recall 0.427, about 2.8 times slower per frame on the lecturer's laptop CPU (the ratio depends on your hardware). Small objects keep their pixels, so more are found, at the cost of time.

On a GPU, or with another version of `ultralytics`, the last digit can differ slightly.